In [1]:
import pandas as pd

In [2]:
from pyspark.sql import SparkSession

In [5]:
import os, sys
os.environ['JAVA_HOME'] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
os.environ['SPARK_HOME'] = "/opt/homebrew/opt/apache-spark/libexec"

In [ ]:
spark = SparkSession.builder.appName("test").getOrCreate()
print("SPARK VERSION :: ",spark.version)

SPARK VERSION ::  4.0.1


25/10/12 22:00:08 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 902977 ms exceeds timeout 120000 ms
25/10/12 22:00:08 WARN SparkContext: Killing executors is not supported by current scheduler.
25/10/12 22:00:15 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [9]:
# Snowflake Client 
# Snowflake Class
import snowflake.connector
import sys 
from typing import List, Dict, Any
from typing import Optional, List, Tuple
# NOTE: We avoid importing PySpark components here to keep the module general.
# The Spark DataFrame (sp_df) is expected to be passed to write_spark_df_to_table.

class SnowflakeClient:
    """
    A client for connecting to and interacting with Snowflake, supporting 
    both direct connection and DataFrame bulk loading (Pandas and Spark).
    """
    def __init__(self, account, user, password, warehouse, database, schema):
        self.account = account
        self.user = user 
        self.password = password 
        self.warehouse = warehouse 
        self.database = database 
        self.schema = schema 
        self.conn = None

    def connect(self):
        """Establishes the connection to Snowflake."""
        try:
            self.conn = snowflake.connector.connect(
                         account = self.account,
                         user = self.user,
                         password = self.password,
                         warehouse = self.warehouse,
                         database = self.database,
                         schema = self.schema    
                    )
            print("Snowflake connection successful.")
        except Exception as e:
            print(f"Snowflake connection failed: {e}", file=sys.stderr)
            self.conn = None


    def close(self):
        """Closes the Snowflake connection."""
        if self.conn: 
            self.conn.close()
            print("Snowflake connection closed.")

    #def execute_query(self, query: str, params: tuple = None) -> List[tuple] | None: 
       

    def execute_query(self, query: str, params: tuple = None) -> Optional[List[Tuple]]:

        """
        Executes a SQL query and returns results if available, 
        committing DML/DDL operations.
        """
        if not self.conn:
            print("Error: Connection not established. Cannot execute query.", file=sys.stderr)
            return None
            
        with self.conn.cursor() as cur:
            try:
                cur.execute(query, params)
                # Ensure DML/DDL like DROP TABLE is committed
                if not self.conn.autocommit:
                    self.conn.commit()
                
                # Try to fetch results, if it's a SELECT query
                try:
                    return cur.fetchall()
                except snowflake.connector.errors.ProgrammingError:
                    return None # No results to fetch (e.g., INSERT, UPDATE, DDL)
            except Exception as e:
                print(f"Error executing query: {query}. Error: {e}", file=sys.stderr)
                return None
            
    def read_data_from_table(self, table_name: str)  -> Optional[List[Tuple]]:
        """Reads all data from a specified table and returns as a list of dicts."""
        if not self.conn:
            print("Error: Connection not established. Cannot read data.", file=sys.stderr)
            return None
            
        query = f"SELECT * FROM {table_name}"
        with self.conn.cursor() as cur:
            cur.execute(query)
            # Fetch column names
            columns = [col[0] for col in cur.description]
            # Fetch data and return as a list of dictionaries
            data = cur.fetchall()
            return [dict(zip(columns, row)) for row in data]
            
    
    # -----------------------------------------------------
    # Spark DataFrame Loader
    # -----------------------------------------------------
    def write_spark_df_to_table(self, table_name: str, sp_df, mode: str = "append"):
        """
        Writes a Spark DataFrame to a Snowflake table using the Spark-Snowflake Connector.

        :param table_name: The name of the Snowflake table.
        :param sp_df: The Spark DataFrame to be written.
        :param mode: Save mode (e.g., 'append', 'overwrite').
        """
        try:
            # Connection properties passed to the Spark Connector
            sfOptions = {
                # sfURL needs to be your full account identifier, which is what 
                # you pass in self.account
                "sfURL": self.account, 
                "sfUser": self.user,
                "sfPassword": self.password,
                "sfWarehouse": self.warehouse,
                "sfDatabase": self.database,
                "sfSchema": self.schema,
                "dbtable": table_name.upper()
            }
            
            print(f"\n--- Starting Spark DF write to table: {table_name}, mode: {mode.lower()} ---")
            
            # --- This is the PySpark write operation using the connector ---
            sp_df.write \
                 .format("snowflake") \
                 .options(**sfOptions) \
                 .mode(mode.lower()) \
                 .save()
            # ---------------------------------------------------------------

            print(f"Spark DataFrame write to {table_name} successful (operation initiated).")

        except Exception as e:
            error_message = f"Error during Spark DataFrame loading. Ensure the 'snowflake' format is available (JAR file required). Error: {e}"
            print(error_message, file=sys.stderr)




In [12]:
#!pip install snowflake-connector-python

In [10]:
from pyspark.sql.types import StructType,StructField
from pyspark.sql.types import * 
import time 
import sys 

# --- 1. SNOWFLAKE CONFIGURATION ---
ACCOUNT = 'TEMTDWR-EY78543'
USER = 'CHINNUNEELA'
PASSWORD = 'Yashwanth14181418'
WAREHOUSE = 'COMPUTE_WH'
DATABASE = 'TEST_DB'
SCHEMA ='TEST_SCHEMA'
SNOWFLAKE_TABLE = "PYSPARK_SAMPLE_DATA"
    
try:
# --- 2. INITIALIZE SPARK SESSION ---
# We are removing the .config() line here because dependency injection is 
# now handled exclusively and reliably by the 'spark-submit --packages' command.
    #spark = SparkSession.builder.appName("SnowflakeSparkDemo").getOrCreate()
    print("\nSpark Session successfully initialized.")

# --- 3. CREATE SPARK DATAFRAME ---
    spark_data = [
    ("Alpha", 100),
    ("Beta", 200),
    ("Gamma", 300)
    ]
    spark_schema = StructType([
    StructField("ITEM_NAME", StringType(), True),
    StructField("VALUE", IntegerType(), True)
    ])

    spark_df = spark.createDataFrame(data=spark_data, schema=spark_schema)
    print("\nSpark DataFrame created:")
    spark_df.printSchema()
    spark_df.show()

# --- 4. INITIALIZE SNOWFLAKE CLIENT & CONNECT ---
    client = SnowflakeClient(ACCOUNT, USER, PASSWORD, WAREHOUSE, DATABASE, SCHEMA)
    client.connect()

    if client.conn:
    # --- 5. WRITE SPARK DATAFRAME TO SNOWFLAKE ---

        options = {
            "sfURL": "https://TEMTDWR-EY78543.snowflakecomputing.com",
            "sfUser": "CHINNUNEELA",
            "sfPassword": "Yashwanth14181418",
            "sfDatabase": "TEST_DB",
            "sfSchema": "TEST_SCHEMA",
            "sfWarehouse": "COMPUTE_WH"
        }

        spark_df.write.format("snowflake").options(**options).option("dbtable", "PYSPARK_SAMPLE_DATA").mode("overwrite").save()
        
        #client.write_spark_df_to_table(
        ##    table_name=SNOWFLAKE_TABLE,
        #    sp_df=spark_df,
        #    mode="overwrite"
        #)

    # Wait a few seconds for the Spark job to finish writing
    time.sleep(5) 

# --- 6. READ DATA BACK FROM SNOWFLAKE (using Python Connector) ---
    print(f"\n--- Reading data back from {SNOWFLAKE_TABLE} via Python Connector ---")
    retrieved_data = client.read_data_from_table(SNOWFLAKE_TABLE)

    if retrieved_data:
        print(f"Successfully retrieved {len(retrieved_data)} rows:")
        for row in retrieved_data:
            print(row)
    else:
        print("Failed to retrieve data or table is empty.")

    # --- 7. CLEANUP ---
    print(f"\nDropping table {SNOWFLAKE_TABLE}.")
    client.execute_query(f"DROP TABLE IF EXISTS {SNOWFLAKE_TABLE}")

    client.close()
    spark.stop()
    print("\nCleanup complete. Spark session stopped.")

except Exception as e:
    print(f"\nAn UNEXPECTED error occurred during the execution: {e}", file=sys.stderr)




Spark Session successfully initialized.

Spark DataFrame created:
root
 |-- ITEM_NAME: string (nullable = true)
 |-- VALUE: integer (nullable = true)

+---------+-----+
|ITEM_NAME|VALUE|
+---------+-----+
|    Alpha|  100|
|     Beta|  200|
|    Gamma|  300|
+---------+-----+

Snowflake connection successful.



An UNEXPECTED error occurred during the execution: An error occurred while calling o67.save.
: org.apache.spark.SparkClassNotFoundException: [DATA_SOURCE_NOT_FOUND] Failed to find the data source: snowflake. Make sure the provider name is correct and the package is properly registered and compatible with your Spark version. SQLSTATE: 42K02
	at org.apache.spark.sql.errors.QueryExecutionErrors$.dataSourceNotFoundError(QueryExecutionErrors.scala:722)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSource(DataSource.scala:681)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSourceV2(DataSource.scala:740)
	at org.apache.spark.sql.classic.DataFrameWriter.lookupV2Provider(DataFrameWriter.scala:626)
	at org.apache.spark.sql.classic.DataFrameWriter.saveInternal(DataFrameWriter.scala:135)
	at org.apache.spark.sql.classic.DataFrameWriter.save(DataFrameWriter.scala:126)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at